# Setup

In [ ]:
from pathlib import Path

import xarray as xr

import cloudy as cd
import ps

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = Client(cluster)

# RAMS

In [ ]:
rams_ds = ps.icons.read_full_rams_output("ic_wk")
# Limit the data variables
rams_ds = cd.rams.calculate_bsr_variables(
    ds=rams_ds, base_state=rams_ds.isel({"time": 0}, drop=False)
)
rams_ds = rams_ds[["UC", "VC", "WC", "THETA_bsr"]]
rams_ds

In [ ]:
# Just pick t=100 minutes as the sample
computed_ds = rams_ds.sel({"time": ps.icons.to_sim_time(100)}).compute()
computed_ds.to_netcdf(
    "/home/cmdavis4/projects/cloudy/examples/example_data/pv_example_rams_data.nc"
)

# Trajectories

In [ ]:
trajectory_ds = xr.open_dataset(
    "/moonbow/cmdavis4/projects/bl_transport/rams_io/isolated_convection_wk/all_tracers/newtracer_forcing-warmbubble4_deltax-500_frqlite-2_nobulku/trajectories/thesisplus_ft.nc"
)
trajectory_ds

In [ ]:
lofted_parcel_ixs = trajectory_ds["parcel_ix"].values[trajectory_ds["lofted"]]
lofted_ds = trajectory_ds[["x", "y", "z", "initialization_time", "THETA"]].sel(
    {"parcel_ix": cd.utils.maybe_random_choice(lofted_parcel_ixs, 25)}
)
lofted_ds = lofted_ds.compute()
lofted_ds

In [ ]:
lofted_ds.to_netcdf(
    "/home/cmdavis4/projects/cloudy/examples/example_data/pv_example_trajectory_data.nc"
)